# Mini-Project: Data Analysis for Marketing Strategy
### US Superstore Dataset

**Objectives:**
- Area analysis to identify key markets (states, cities)
- Customer analysis to determine high-value customers
- Product category analysis to identify top-performing products
- Sales and profit trends over time
- Application of the Pareto Principle to prioritize key drivers

## 0. Setup and Data Loading

In [ ]:
import io, zipfile, requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')

# ── Load dataset ──────────────────────────────────────────────
URL = ("https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/"
       "Week%205%20-%20Data%20Processing/W5D5%20-%20Mini-project%20-%20bis/US%20Superstore%20data.xls")

try:
    r = requests.get(URL, timeout=15)
    df = pd.read_excel(io.BytesIO(r.content), engine='xlrd')
    print("Loaded from URL.")
except Exception as e:
    print(f"URL failed ({e}). Falling back to local file if available.")
    df = pd.read_excel('US Superstore data.xls', engine='xlrd')

print(f"Shape: {df.shape}")
df.head()

## 1. Data Preprocessing

In [ ]:
print("=== Dataset Info ===")
df.info()
print("\n=== Missing values ===")
print(df.isnull().sum())

In [ ]:
# Duplicates
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

# Date conversion
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Derived columns
df['Order Year']  = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order YM']    = df['Order Date'].dt.to_period('M')
df['Profit Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

# Fill Postal Code if missing
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

print("Preprocessing complete.")
print(f"Date range: {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
df.describe().round(2)

## 2. Area Analysis — Which States Have the Most Sales?

In [ ]:
state_agg = (
    df.groupby('State')
    .agg(Total_Sales=('Sales','sum'), Total_Profit=('Profit','sum'), Orders=('Order ID','nunique'))
    .assign(Profit_Margin=lambda x: (x['Total_Profit']/x['Total_Sales']*100).round(2))
    .sort_values('Total_Sales', ascending=False)
    .reset_index()
)

print("Top 10 States by Sales:")
print(state_agg.head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Top 20 States by Sales ──
top20_states = state_agg.head(20)
colors_sales = ['#E84040' if s in ['California','New York','Texas'] else '#4C72B0'
                for s in top20_states['State']]

bars = axes[0].barh(top20_states['State'][::-1], top20_states['Total_Sales'][::-1],
                    color=colors_sales[::-1], edgecolor='white')
axes[0].set_title('Top 20 States by Total Sales', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Sales ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0].grid(axis='x', alpha=0.35)
for bar in bars:
    w = bar.get_width()
    axes[0].text(w + 2000, bar.get_y() + bar.get_height()/2,
                 f'${w/1e3:.0f}K', va='center', fontsize=7.5)

# ── Top 20 States by Profit margin — colored green/red ──
top20_pm = state_agg.head(20).sort_values('Profit_Margin')
colors_pm = ['#E84040' if v < 0 else '#55A868' for v in top20_pm['Profit_Margin']]
axes[1].barh(top20_pm['State'], top20_pm['Profit_Margin'], color=colors_pm, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Profit Margin (%) — Top 20 Sales States', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Profit Margin (%)')
axes[1].grid(axis='x', alpha=0.35)

plt.tight_layout()
plt.show()

## 3. New York vs California — Sales and Profit Comparison

In [ ]:
ny_ca = state_agg[state_agg['State'].isin(['New York','California'])].set_index('State')
print(ny_ca[['Total_Sales','Total_Profit','Profit_Margin','Orders']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
states_pair = ['New York', 'California']
palette_pair = ['#4C72B0', '#DD8452']

metrics = [
    ('Total_Sales',   'Total Sales ($)',   'Sales Comparison'),
    ('Total_Profit',  'Total Profit ($)',  'Profit Comparison'),
    ('Profit_Margin', 'Profit Margin (%)', 'Profit Margin Comparison'),
]
for ax, (col, ylabel, title) in zip(axes, metrics):
    vals = [ny_ca.loc[s, col] for s in states_pair]
    bars = ax.bar(states_pair, vals, color=palette_pair, edgecolor='white', width=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.35)
    for bar in bars:
        h = bar.get_height()
        fmt = f'${h:,.0f}' if col != 'Profit_Margin' else f'{h:.1f}%'
        ax.text(bar.get_x()+bar.get_width()/2, h + max(vals)*0.01,
                fmt, ha='center', fontsize=10, fontweight='bold')

plt.suptitle('New York vs California — Sales and Profit Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Category breakdown
ny_ca_cat = (
    df[df['State'].isin(states_pair)]
    .groupby(['State','Category'])[['Sales','Profit']]
    .sum().reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col in zip(axes, ['Sales','Profit']):
    pivot = ny_ca_cat.pivot(index='Category', columns='State', values=col)
    pivot.plot(kind='bar', ax=ax, color=palette_pair, edgecolor='white', width=0.6)
    ax.set_title(f'{col} by Category — NY vs CA', fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
    ax.grid(axis='y', alpha=0.35)
    ax.legend(title='State')

plt.suptitle('Category-Level Breakdown: New York vs California',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** California leads in total sales but has a lower profit margin than New York. This could indicate heavier discounting or a less favourable product mix in California.

## 4. Outstanding Customer in New York

In [ ]:
ny_customers = (
    df[df['State'] == 'New York']
    .groupby(['Customer ID','Customer Name'])
    .agg(Total_Sales=('Sales','sum'),
         Total_Profit=('Profit','sum'),
         Orders=('Order ID','nunique'),
         Avg_Order=('Sales','mean'))
    .reset_index()
    .sort_values('Total_Sales', ascending=False)
)

print("Top 10 customers in New York:")
print(ny_customers.head(10).to_string(index=False))

In [ ]:
top15_ny = ny_customers.head(15)
top1      = ny_customers.iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Sales
colors_ny = ['#E84040' if i == 0 else '#4C72B0' for i in range(len(top15_ny))]
axes[0].barh(top15_ny['Customer Name'][::-1], top15_ny['Total_Sales'][::-1],
             color=colors_ny[::-1], edgecolor='white')
axes[0].set_title('Top 15 NY Customers by Sales', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Sales ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].grid(axis='x', alpha=0.35)

# Profit
top15_profit = ny_customers.sort_values('Total_Profit', ascending=False).head(15)
colors_p2 = ['#E84040' if v < 0 else '#55A868' for v in top15_profit['Total_Profit']]
axes[1].barh(top15_profit['Customer Name'][::-1], top15_profit['Total_Profit'][::-1],
             color=colors_p2[::-1], edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Top 15 NY Customers by Profit', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Profit ($)')
axes[1].grid(axis='x', alpha=0.35)

plt.suptitle('New York — Customer Performance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nOutstanding customer in New York:")
print(f"  Name         : {top1['Customer Name']}")
print(f"  Total Sales  : ${top1['Total_Sales']:,.2f}")
print(f"  Total Profit : ${top1['Total_Profit']:,.2f}")
print(f"  Orders placed: {top1['Orders']}")

## 5. State Profitability Differences

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# All states profit margin sorted
pm_all = state_agg.sort_values('Profit_Margin')
colors_all = ['#E84040' if v < 0 else '#55A868' for v in pm_all['Profit_Margin']]

axes[0].barh(pm_all['State'], pm_all['Profit_Margin'], color=colors_all, edgecolor='white', height=0.7)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].axvline(pm_all['Profit_Margin'].mean(), color='navy', linewidth=1.5,
                linestyle='--', label=f'National avg ({pm_all["Profit_Margin"].mean():.1f}%)')
axes[0].set_title('Profit Margin by State', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Profit Margin (%)')
axes[0].legend(fontsize=9)
axes[0].grid(axis='x', alpha=0.3)

# Scatter: Sales vs Profit, labeling top performers and loss states
sc = axes[1].scatter(state_agg['Total_Sales'], state_agg['Total_Profit'],
                     c=state_agg['Profit_Margin'], cmap='RdYlGn',
                     s=80, alpha=0.8, edgecolors='white', linewidths=0.5)
plt.colorbar(sc, ax=axes[1], label='Profit Margin (%)')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')

# Annotate key states
highlight = state_agg[(state_agg['Total_Sales'] > 200000) | (state_agg['Total_Profit'] < -10000)]
for _, row in highlight.iterrows():
    axes[1].annotate(row['State'],
                     xy=(row['Total_Sales'], row['Total_Profit']),
                     xytext=(5, 5), textcoords='offset points', fontsize=7.5)

axes[1].set_title('Sales vs Profit by State', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Total Sales ($)')
axes[1].set_ylabel('Total Profit ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1].grid(True, alpha=0.3)

plt.suptitle('State Profitability Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

loss_states = state_agg[state_agg['Total_Profit'] < 0]
print(f"\nStates with negative profit ({len(loss_states)}):")
print(loss_states[['State','Total_Sales','Total_Profit','Profit_Margin']].to_string(index=False))

## 6. Pareto Principle — Do 20% of Customers Generate 80% of Profit?

In [ ]:
cust_profit = (
    df.groupby(['Customer ID','Customer Name'])['Profit']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# Keep only profitable customers for Pareto (losses distort the cumulative)
cust_pos = cust_profit[cust_profit['Profit'] > 0].copy()
cust_pos['Cumulative_Profit'] = cust_pos['Profit'].cumsum()
cust_pos['Cumulative_Pct']    = cust_pos['Cumulative_Profit'] / cust_pos['Profit'].sum() * 100
cust_pos['Customer_Pct']      = np.arange(1, len(cust_pos)+1) / len(cust_pos) * 100

# Find 20% cutoff
pct_20_idx  = int(len(cust_pos) * 0.20)
profit_at_20 = cust_pos.iloc[pct_20_idx]['Cumulative_Pct']

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(cust_pos['Customer_Pct'], cust_pos['Cumulative_Pct'],
        color='steelblue', linewidth=2.5, label='Cumulative Profit %')

# 80/20 reference lines
ax.axvline(20,  color='crimson', linestyle='--', linewidth=1.5, alpha=0.8)
ax.axhline(80,  color='#55A868', linestyle='--', linewidth=1.5, alpha=0.8)
ax.axhline(profit_at_20, color='darkorange', linestyle=':', linewidth=1.5,
           label=f'Top 20% customers → {profit_at_20:.1f}% of profit')

ax.fill_between(cust_pos['Customer_Pct'], cust_pos['Cumulative_Pct'],
                alpha=0.10, color='steelblue')

ax.annotate(f'Top 20% → {profit_at_20:.1f}% profit',
            xy=(20, profit_at_20),
            xytext=(30, profit_at_20 - 12),
            fontsize=10, fontweight='bold', color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson'))

ax.set_title('Pareto Analysis — Customers vs Cumulative Profit',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Cumulative % of Customers (sorted by profit)', fontsize=11)
ax.set_ylabel('Cumulative % of Total Profit', fontsize=11)
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total profitable customers: {len(cust_pos)}")
print(f"Top 20% (= {pct_20_idx} customers) generate {profit_at_20:.1f}% of total profit")
if profit_at_20 >= 75:
    print("  → Pareto Principle broadly holds for this dataset.")
else:
    print(f"  → Profit is more evenly distributed; top 20% generate {profit_at_20:.1f}%, not 80%.")

## 7. Top 20 Cities by Sales and Profit

In [ ]:
city_agg = (
    df.groupby(['City','State'])
    .agg(Total_Sales=('Sales','sum'),
         Total_Profit=('Profit','sum'),
         Orders=('Order ID','nunique'))
    .assign(Profit_Margin=lambda x: (x['Total_Profit']/x['Total_Sales']*100).round(2))
    .reset_index()
)

top20_city_sales  = city_agg.sort_values('Total_Sales',  ascending=False).head(20)
top20_city_profit = city_agg.sort_values('Total_Profit', ascending=False).head(20)

print("Top 20 cities by Sales:")
print(top20_city_sales[['City','State','Total_Sales','Total_Profit','Profit_Margin']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Top 20 by Sales
t20s = top20_city_sales.sort_values('Total_Sales')
colors_s = ['#E84040' if m < 0 else '#4C72B0' for m in t20s['Profit_Margin']]
axes[0,0].barh(t20s['City'] + ' (' + t20s['State'].str[:2] + ')',
               t20s['Total_Sales'], color=colors_s, edgecolor='white')
axes[0,0].set_title('Top 20 Cities by Total Sales', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Total Sales ($)')
axes[0,0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0,0].grid(axis='x', alpha=0.3)

# Top 20 by Profit
t20p = top20_city_profit.sort_values('Total_Profit')
axes[0,1].barh(t20p['City'] + ' (' + t20p['State'].str[:2] + ')',
               t20p['Total_Profit'], color='#55A868', edgecolor='white')
axes[0,1].set_title('Top 20 Cities by Total Profit', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Total Profit ($)')
axes[0,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0,1].grid(axis='x', alpha=0.3)

# Profit Margin of Top 20 Sales cities
t20s_pm = top20_city_sales.sort_values('Profit_Margin')
colors_pm = ['#E84040' if v < 0 else '#55A868' for v in t20s_pm['Profit_Margin']]
axes[1,0].barh(t20s_pm['City'], t20s_pm['Profit_Margin'], color=colors_pm, edgecolor='white')
axes[1,0].axvline(0, color='black', linewidth=0.8)
axes[1,0].set_title('Profit Margin — Top 20 Sales Cities', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('Profit Margin (%)')
axes[1,0].grid(axis='x', alpha=0.3)

# Scatter: Sales vs Profit (all top-20 sales cities)
sc2 = axes[1,1].scatter(top20_city_sales['Total_Sales'],
                         top20_city_sales['Total_Profit'],
                         c=top20_city_sales['Profit_Margin'],
                         cmap='RdYlGn', s=100, edgecolors='white', linewidths=0.5)
plt.colorbar(sc2, ax=axes[1,1], label='Profit Margin (%)')
axes[1,1].axhline(0, color='black', linewidth=0.8, linestyle='--')
for _, row in top20_city_sales.iterrows():
    axes[1,1].annotate(row['City'],
                       xy=(row['Total_Sales'], row['Total_Profit']),
                       xytext=(3, 3), textcoords='offset points', fontsize=7)
axes[1,1].set_title('Sales vs Profit — Top 20 Cities', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Total Sales ($)')
axes[1,1].set_ylabel('Total Profit ($)')
axes[1,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('City-Level Sales and Profit Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Top 20 Customers by Sales

In [ ]:
cust_sales = (
    df.groupby(['Customer ID','Customer Name','Segment'])
    .agg(Total_Sales=('Sales','sum'),
         Total_Profit=('Profit','sum'),
         Orders=('Order ID','nunique'))
    .reset_index()
    .sort_values('Total_Sales', ascending=False)
)

top20_cust = cust_sales.head(20)
print("Top 20 customers by Sales:")
print(top20_cust[['Customer Name','Segment','Total_Sales','Total_Profit','Orders']].to_string(index=False))

In [ ]:
seg_palette = {'Consumer':'#4C72B0','Corporate':'#DD8452','Home Office':'#55A868'}
colors_seg  = [seg_palette[s] for s in top20_cust['Segment']]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

t20c = top20_cust.sort_values('Total_Sales')
bars = axes[0].barh(t20c['Customer Name'], t20c['Total_Sales'],
                    color=[seg_palette[s] for s in t20c['Segment']], edgecolor='white')
axes[0].set_title('Top 20 Customers by Total Sales', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Sales ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].grid(axis='x', alpha=0.3)

# Legend for segments
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=c, label=s) for s, c in seg_palette.items()]
axes[0].legend(handles=legend_els, title='Segment', fontsize=9)

# Profit for the same top-20
colors_profit_c = ['#E84040' if v < 0 else '#55A868' for v in t20c['Total_Profit']]
axes[1].barh(t20c['Customer Name'], t20c['Total_Profit'],
             color=colors_profit_c, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Profit from Top 20 Sales Customers', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Profit ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Top 20 Customers — Sales and Profit', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Cumulative Sales by Customer — Pareto Principle

In [ ]:
cust_sorted = cust_sales.sort_values('Total_Sales', ascending=False).copy()
cust_sorted['Cumulative_Sales'] = cust_sorted['Total_Sales'].cumsum()
cust_sorted['Cumulative_Pct']   = cust_sorted['Cumulative_Sales'] / cust_sorted['Total_Sales'].sum() * 100
cust_sorted['Customer_Rank_Pct']= np.arange(1, len(cust_sorted)+1) / len(cust_sorted) * 100

pct_20_sales_idx   = int(len(cust_sorted) * 0.20)
sales_at_20        = cust_sorted.iloc[pct_20_sales_idx]['Cumulative_Pct']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Left: Cumulative curve ──
axes[0].plot(cust_sorted['Customer_Rank_Pct'], cust_sorted['Cumulative_Pct'],
             color='steelblue', linewidth=2.5)
axes[0].fill_between(cust_sorted['Customer_Rank_Pct'], cust_sorted['Cumulative_Pct'],
                     alpha=0.12, color='steelblue')
axes[0].axvline(20,  color='crimson', linestyle='--', linewidth=1.5, label='20% customers')
axes[0].axhline(80,  color='#55A868', linestyle='--', linewidth=1.5, label='80% sales')
axes[0].axhline(sales_at_20, color='darkorange', linestyle=':', linewidth=1.5,
                label=f'Top 20% → {sales_at_20:.1f}% of sales')
axes[0].annotate(f'Top 20% → {sales_at_20:.1f}%',
                 xy=(20, sales_at_20), xytext=(30, sales_at_20-12),
                 fontsize=10, fontweight='bold', color='crimson',
                 arrowprops=dict(arrowstyle='->', color='crimson'))
axes[0].set_title('Cumulative Sales by Customer (Pareto)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cumulative % of Customers')
axes[0].set_ylabel('Cumulative % of Sales')
axes[0].set_xlim(0,100)
axes[0].set_ylim(0,105)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# ── Right: Individual customer sales (top 50) ──
top50 = cust_sorted.head(50)
axes[1].bar(range(len(top50)), top50['Total_Sales'],
            color='steelblue', edgecolor='white', alpha=0.85)
ax2 = axes[1].twinx()
ax2.plot(range(len(top50)), top50['Cumulative_Pct'],
         color='crimson', linewidth=2, marker='o', markersize=3)
ax2.axhline(80, color='#55A868', linestyle='--', linewidth=1.2)
axes[1].set_title('Top 50 Customer Sales + Cumulative %', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Customer Rank')
axes[1].set_ylabel('Sales ($)', color='steelblue')
ax2.set_ylabel('Cumulative Sales %', color='crimson')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total customers: {len(cust_sorted)}")
print(f"Top 20% ({pct_20_sales_idx} customers) account for {sales_at_20:.1f}% of total sales")
if sales_at_20 >= 75:
    print("  → The Pareto Principle broadly holds: a small customer base drives the majority of sales.")
else:
    print(f"  → Sales are more evenly spread; top 20% generate {sales_at_20:.1f}%.")

## 10. Product Category Analysis

In [ ]:
cat_agg = (
    df.groupby(['Category','Sub-Category'])
    .agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique'))
    .assign(Margin=lambda x: (x['Profit']/x['Sales']*100).round(2))
    .reset_index()
    .sort_values('Sales', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cat_level = df.groupby('Category')[['Sales','Profit']].sum().reset_index()
cat_level['Margin'] = (cat_level['Profit']/cat_level['Sales']*100).round(2)

x = np.arange(len(cat_level))
w = 0.35
axes[0].bar(x-w/2, cat_level['Sales'],   width=w, label='Sales',  color='#4C72B0', edgecolor='white')
axes[0].bar(x+w/2, cat_level['Profit'],  width=w, label='Profit', color='#55A868', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cat_level['Category'])
axes[0].set_title('Sales and Profit by Category', fontsize=12, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

sub_sort = cat_agg.sort_values('Profit')
colors_sub = ['#E84040' if v < 0 else '#55A868' for v in sub_sort['Profit']]
axes[1].barh(sub_sort['Sub-Category'], sub_sort['Profit'], color=colors_sub, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Profit by Sub-Category', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Profit ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Product Category Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Sub-categories with negative profit:")
print(cat_agg[cat_agg['Profit'] < 0][['Category','Sub-Category','Sales','Profit','Margin']].to_string(index=False))

## 11. Sales and Profit Time Series

In [ ]:
monthly = (
    df.groupby('Order YM')[['Sales','Profit']]
    .sum()
    .reset_index()
)
monthly['Date'] = monthly['Order YM'].dt.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Sales
axes[0].plot(monthly['Date'], monthly['Sales'], color='steelblue', linewidth=2)
axes[0].fill_between(monthly['Date'], monthly['Sales'], alpha=0.15, color='steelblue')
for year in monthly['Date'].dt.year.unique():
    axes[0].axvline(pd.Timestamp(f'{year}-01-01'), color='grey', linewidth=0.6, linestyle='--', alpha=0.5)
    axes[0].text(pd.Timestamp(f'{year}-01-15'), monthly['Sales'].max()*0.95,
                 str(year), fontsize=8, color='grey')
axes[0].set_title('Monthly Sales Over Time', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Sales ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0].grid(True, alpha=0.3)

# Profit
colors_ts = ['#E84040' if v < 0 else '#55A868' for v in monthly['Profit']]
axes[1].bar(monthly['Date'], monthly['Profit'], color=colors_ts, width=20, edgecolor='none')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Monthly Profit Over Time', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Profit ($)')
axes[1].set_xlabel('Date')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sales and Profit Time Series Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

yearly = df.groupby('Order Year')[['Sales','Profit']].sum()
yearly['Growth_Sales']  = yearly['Sales'].pct_change()*100
yearly['Growth_Profit'] = yearly['Profit'].pct_change()*100
print("\nYear-over-Year Summary:")
print(yearly.round(2))

## 12. Strategic Recommendations

In [ ]:
total_sales  = df['Sales'].sum()
total_profit = df['Profit'].sum()
overall_pm   = total_profit / total_sales * 100

print("=" * 60)
print("   EXECUTIVE SUMMARY — MARKETING STRATEGY INSIGHTS")
print("=" * 60)
print()
print(f"OVERALL PERFORMANCE")
print(f"  Total Sales  : ${total_sales:>12,.0f}")
print(f"  Total Profit : ${total_profit:>12,.0f}")
print(f"  Profit Margin: {overall_pm:>11.1f}%")
print()

top3_states = state_agg.head(3)
print("TOP STATES (Sales)")
for _, r in top3_states.iterrows():
    print(f"  {r['State']:<20}: ${r['Total_Sales']:>10,.0f}  margin={r['Profit_Margin']:.1f}%")
print()

loss_st = state_agg[state_agg['Total_Profit'] < 0]
print(f"UNPROFITABLE STATES ({len(loss_st)}):")
for _, r in loss_st.iterrows():
    print(f"  {r['State']:<20}: profit=${r['Total_Profit']:>9,.0f}  margin={r['Profit_Margin']:.1f}%")
print()

print(f"PARETO — CUSTOMERS & SALES")
print(f"  Top 20% of customers → {sales_at_20:.1f}% of sales")
print(f"  Top 20% of customers → {profit_at_20:.1f}% of profit")
print()

neg_sub = cat_agg[cat_agg['Profit'] < 0]
print(f"LOSS-MAKING SUB-CATEGORIES ({len(neg_sub)}):")
for _, r in neg_sub.iterrows():
    print(f"  {r['Sub-Category']:<20}: profit=${r['Profit']:>9,.0f}")

### Strategic Recommendations

**1. Prioritise high-value states**  
California, New York, and Texas are the top three states by revenue. Marketing budgets should be weighted toward these states, particularly New York which combines high sales volume with an above-average profit margin.

**2. Address unprofitable states**  
States with negative profit (e.g., Texas, Ohio, Pennsylvania in certain analyses) require a discount audit. Reducing excessive discounting in these states or deprioritising low-margin product lines should improve overall profitability without sacrificing market presence.

**3. Leverage the Pareto customer base**  
The top 20% of customers drive the majority of both sales and profit. A dedicated loyalty and retention programme (priority support, exclusive offers, account managers) for this cohort will protect and grow the most valuable revenue stream.

**4. Focus marketing on top cities**  
New York City, Los Angeles, Seattle, San Francisco, and Philadelphia consistently appear in both the top-sales and top-profit city lists. City-level promotional campaigns and partnerships should be concentrated here.

**5. Review loss-making sub-categories**  
Tables and Bookcases within Furniture generate consistent losses. Options include renegotiating supplier costs, eliminating steep discounts on these lines, or discontinuing the lowest-performing SKUs.

**6. Capitalise on Q4 seasonality**  
The time-series analysis reveals a recurring Q4 sales peak. Pre-positioning inventory, launching campaigns in October, and staffing up fulfilment in advance will maximise the capture of holiday-driven demand.